In [ ]:
import polars as pl


ImportError: attempted relative import with no known parent package

In [3]:
def read_and_save_regio_categories(path_excel: str, 
                                 save_path:str = 'src/REGIONALIZATION/ref_data/categories/', 
                                 header_row:int=1):
    overview = pl.read_excel(path_excel,
                            columns= [0, 1, 2, 3],
                            read_options={"header_row": header_row}
                        )

    return overview

In [6]:
overview = read_and_save_regio_categories('/home/307920@ontw.alfa.local/projects/epn-ma-master/src/REGIONALIZATION/ref_data/categories/Regionalization_categories_overview_copy.xlsx')

In [7]:
overview_2 = overview.filter(pl.col('Category').str.starts_with('Industry'))

In [ ]:
overview_2

Energy carrier,Type,Sector,Category
str,str,str,str
"""Electricity""","""DSO""","""Demand""","""Industry_aluminium"""
"""Electricity""","""DSO""","""Demand""","""Industry_chemicals"""
"""Electricity""","""TSO""","""Demand""","""Industry_chemicals"""
"""Electricity""","""DSO""","""Demand""","""Industry_food"""
"""Electricity""","""TSO""","""Demand""","""Industry_fuels"""
…,…,…,…
"""Methane""","""TSO""","""Demand""","""Industry_refineries"""
"""Methane""","""TSO""","""Demand""","""Industry_steel"""
"""Methane""","""DSO""","""Supply""","""Industry_chemicals"""


In [7]:
from geopy.geocoders import Nominatim


geolocator = Nominatim(user_agent='regionalization')

location = geolocator.geocode('Tata Steel NL')

if location:
    print(f"Latitude: {location.latitude}")
    print(f"Longitude: {location.longitude}")
else:
    print("Location not found")




Latitude: 52.4845549
Longitude: 4.6035028


Pull from ETM

In [3]:
from pyetm import Session


def fetch_power_to_heat(session_id: str | int, gqueries: list[str] | None = None) -> dict:
    """
    Load an ETM session and fetch power-to-heat gquery results.

    CAVEAT: the default gqueries below are the only PtH-related keys I could
    confirm actually exist — both are household/district-heating specific, NOT
    an industry-sector figure. Before trusting this for the regionalization
    PtH gap, confirm the right key(s) via ETM's inspect tool:
        https://engine.energytransitionmodel.com/inspect/<session_id>/gqueries
    which lists every gquery available for that specific session -- search it
    for "p2h" or "power_to_heat" to find the industry-level equivalent.
    """
    if gqueries is None:
        gqueries = [
            "households_flexibility_p2h_electricity_capacity",
            "energy_heat_flexibility_p2h_heatpump_mt_electricity_capacity",
        ]

    session = Session.load(int(session_id))
    session.add_queries(gqueries)
    session.execute_queries()

    return session.results(columns=["present", "future"]).to_dict()


def fetch_power_to_heat_all_sessions(
    etm_sessions: dict[tuple[str, str], str],  # {(scenario, year): etm_session_id}
    gqueries: list[str] | None = None,
) -> dict:
    """Fetch PtH gquery results for every (scenario, year) -> ETM session."""
    results = {}
    for (scenario, year), session_id in etm_sessions.items():
        try:
            results[(scenario, year)] = fetch_power_to_heat(session_id, gqueries=gqueries)
        except Exception as e:
            print(f"[ERROR] {scenario}/{year} (session {session_id}): {e}")
            results[(scenario, year)] = None
    return results

In [4]:
from pyetm import Session


def fetch_power_to_heat(session_id: str | int, gquery_key: str) -> dict:
    """Load a live ETM session and fetch one gquery result."""
    session = Session.load(int(session_id))
    session.add_queries([gquery_key])
    session.execute_queries()

    return session.results(columns=["present", "future"]).to_dict()

In [11]:
fetch_power_to_heat(session_id='1450934', gquery_key='power_to_heat')

Warnings:
  results:
    [WARNING] Gquery power_to_heat does not exist
Warnings:
  results:
    [WARNING] Gquery power_to_heat does not exist
  - Gquery power_to_heat does not exist


{}

In [13]:
import requests


def find_gqueries(search_term: str) -> list[dict]:
    """
    Fetch every gquery from ETM and return ones whose key, description, or
    labels contain `search_term` (case-insensitive).
    """
    response = requests.get("https://engine.energytransitionmodel.com/api/v3/gqueries")
    response.raise_for_status()
    all_gqueries = response.json()

    term = search_term.lower()
    return [
        gq for gq in all_gqueries
        if term in gq["key"].lower()
        or (gq.get("description") or "").lower().find(term) != -1
        or any(term in label.lower() for label in gq.get("labels", []))
    ]


for gq in find_gqueries("power_to_heat"):
    print(gq["key"], "-", gq["unit"], "-", gq.get("description"))

electricity_demand_flexible_power_to_heat_agriculture_capacity - MW - Selects the electricity input capacity of power-to-heat technologies for agriculture using the p2h node group. Capacity of demand technologies is considered negative for the installed flexible capacities chart.
electricity_demand_flexible_power_to_heat_capacity - MW - Selects the electricity input capacity of power-to-heat technologies using the p2h node group. Capacity of demand technologies is considered negative for the installed flexible capacities chart.
electricity_demand_flexible_power_to_heat_district_heating_capacity - MW - Selects the electricity input capacity of power-to-heat technologies for district heating using the p2h node group. Capacity of demand technologies is considered negative for the installed flexible capacities chart.
electricity_demand_flexible_power_to_heat_industry_capacity - MW - Selects the electricity input capacity of power-to-heat technologies for industry using the p2h node group. 